In [ ]:
#Load the trained model for the image-matching stage and run it from the image-matching directory, not from the current directory.
import os
from copy import deepcopy
import torch
import cv2
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from src.utils.plotting import make_matching_figure
from src.loftr import LoFTR, default_cfg
_default_cfg = deepcopy(default_cfg)
_default_cfg['coarse']['temp_bug_fix'] =True # set to False when using the old ckpt
matcher = LoFTR(config=_default_cfg)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
state_dict_path = ""
state_dict = torch.load(state_dict_path, map_location=device)
matcher.load_state_dict(state_dict)
matcher = matcher.eval().to(device)

In [ ]:
#Using the image-matching model, 
#crack annotations from clear images are transferred to degraded images to obtain crack candidate regions in the degraded images.

import os
import cv2
import numpy as np
import torch
from scipy.spatial import cKDTree
from skimage import morphology
from scipy.interpolate import Rbf
# ===================== Parameters =====================
TILE = 32
H, W = 480, 480
CONF_TH = 0.2
X_AXIS_THRE, Y_AXIS_THRE = 60, 60
device = "cuda" if torch.cuda.is_available() else "cpu"

# ===================== Dataset paths =====================
clear_image_dir    = "dataset/preprocessing/clear_image"
degraded_image_dir = "dataset/preprocessing/degrade_image"
clear_label_dir    = "dataset/preprocessing/clear_image_label"

output_root = "dataset/result"

# ===================== Output folders =====================
# transferred crack labels from clear images
out_transferred_clear_labels = os.path.join(output_root, "transferred_clear_labels")

# candidate crack regions on degraded images
out_degraded_candidate_regions = os.path.join(output_root, "degraded_candidate_regions")

# visualization of degraded candidate regions
out_degraded_candidate_visualization = os.path.join(output_root, "degraded_candidate_visualization")

# transferred degraded images (background removed outside transferred area)
out_transferred_degraded_images = os.path.join(output_root, "transferred_degraded_images")

for d in [
    out_transferred_clear_labels,
    out_degraded_candidate_regions,
    out_degraded_candidate_visualization,
    out_transferred_degraded_images,
]:
    os.makedirs(d, exist_ok=True)

# ===================== Unicode image I/O =====================
def imread_unicode(path, flags=cv2.IMREAD_COLOR):
    data = np.fromfile(path, dtype=np.uint8)
    return cv2.imdecode(data, flags)

def imwrite_unicode(path, img):
    ext = os.path.splitext(path)[1]
    ok, buf = cv2.imencode(ext, img)
    if ok:
        buf.tofile(path)

# ===================== Utility functions =====================
def infer_center_offset(coords, tile):
    mods = np.mod(coords, tile)
    off = float(np.median(mods))
    if abs(off) < 1.5 or abs(off - tile) < 1.5:
        off = 0.0
    return off

def center_to_index(c, tile, offset):
    return int(round((float(c) - offset) / tile))

def idx_bounds(k, tile, limit):
    return max(0, k * tile), min(limit, (k + 1) * tile)

def copy_region(src, dst, sx0, sx1, sy0, sy1, dx0, dx1, dy0, dy1, filled):
    if sx1 <= sx0 or sy1 <= sy0 or dx1 <= dx0 or dy1 <= dy0:
        return
    h = min(sy1 - sy0, dy1 - dy0)
    w = min(sx1 - sx0, dx1 - dx0)
    mask = filled[dy0:dy0+h, dx0:dx0+w]
    blk = src[sy0:sy0+h, sx0:sx0+w]
    dst[dy0:dy0+h, dx0:dx0+w] = np.where(mask, dst[dy0:dy0+h, dx0:dx0+w], blk)
    filled[dy0:dy0+h, dx0:dx0+w] = True

def coarse_to_pixel(mk0, mk1):
    Hc, Wc = H // TILE, W // TILE
    c0, r0 = mk0 % Wc, mk0 // Wc
    c1, r1 = mk1 % Wc, mk1 // Wc
    p0 = np.stack([(c0 + 0.5) * TILE, (r0 + 0.5) * TILE], 1)
    p1 = np.stack([(c1 + 0.5) * TILE, (r1 + 0.5) * TILE], 1)
    return p0, p1

def build_inverse_rbf(dst_pts, src_pts):
    Fx = Rbf(dst_pts[:,0], dst_pts[:,1], src_pts[:,0], function="thin_plate")
    Fy = Rbf(dst_pts[:,0], dst_pts[:,1], src_pts[:,1], function="thin_plate")
    return Fx, Fy

# ===================== Main loop =====================
names = sorted([f for f in os.listdir(degraded_image_dir) if f.lower().endswith(".jpg")])

for fname in names:

    img0_path = os.path.join(clear_image_dir, fname)
    img1_path = os.path.join(degraded_image_dir, fname)
    mask_path = os.path.join(clear_label_dir, fname)

    if not (os.path.exists(img0_path) and os.path.exists(mask_path)):
        continue

    img0_raw  = imread_unicode(img0_path, cv2.IMREAD_GRAYSCALE)
    img1_raw  = imread_unicode(img1_path, cv2.IMREAD_GRAYSCALE)
    img00_raw = imread_unicode(img0_path, cv2.IMREAD_COLOR)
    img11_raw = imread_unicode(img1_path, cv2.IMREAD_COLOR)
    src_mask  = imread_unicode(mask_path, cv2.IMREAD_GRAYSCALE)

    if any(x is None for x in [img0_raw, img1_raw, img00_raw, img11_raw, src_mask]):
        continue

    img0_raw = cv2.resize(img0_raw, (W, H))
    img1_raw = cv2.resize(img1_raw, (W, H))
    src_mask = cv2.resize(src_mask, (W, H))
    src_mask = (src_mask > 127).astype(np.uint8) * 255

    # ===================== LoFTR matching =====================
    batch = {
        "image0": torch.from_numpy(img0_raw)[None,None].float().to(device)/255,
        "image1": torch.from_numpy(img1_raw)[None,None].float().to(device)/255,
    }

    with torch.no_grad():
        matcher(batch)

    mk0 = batch["i_ids"].cpu().numpy()
    mk1 = batch["j_ids"].cpu().numpy()
    conf = batch["mconf"].cpu().numpy()

    keep = conf >= CONF_TH
    mk0, mk1 = mk0[keep], mk1[keep]
    if len(mk0) < 50:
        continue

    p0, p1 = coarse_to_pixel(mk0, mk1)
    dif = p0 - p1
    dx, dy = dif[:,0].mean(), dif[:,1].mean()
    ok = (np.abs(dif[:,0] - dx) < X_AXIS_THRE) & (np.abs(dif[:,1] - dy) < Y_AXIS_THRE)
    src_pts, dst_pts = p0[ok], p1[ok]
    if len(src_pts) < 40:
        continue

    # ===================== Three-stage label transfer =====================
    out_mask = np.zeros((H, W), np.uint8)
    filled = np.zeros((H, W), bool)

    off_x_s = infer_center_offset(src_pts[:,0], TILE)
    off_y_s = infer_center_offset(src_pts[:,1], TILE)
    off_x_d = infer_center_offset(dst_pts[:,0], TILE)
    off_y_d = infer_center_offset(dst_pts[:,1], TILE)

    # Stage 1 — direct tile copy
    for (xs,ys),(xd,yd) in zip(src_pts, dst_pts):
        ksx, ksy = center_to_index(xs,TILE,off_x_s), center_to_index(ys,TILE,off_y_s)
        kdx, kdy = center_to_index(xd,TILE,off_x_d), center_to_index(yd,TILE,off_y_d)
        copy_region(src_mask,out_mask,
                    *idx_bounds(ksx,TILE,W),*idx_bounds(ksy,TILE,H),
                    *idx_bounds(kdx,TILE,W),*idx_bounds(kdy,TILE,H),
                    filled)

    # Stage 2 — nearest neighbour tile propagation
    dst_centers, delta_idx = [], []
    for (xs,ys),(xd,yd) in zip(src_pts,dst_pts):
        ksx, ksy = center_to_index(xs,TILE,off_x_s), center_to_index(ys,TILE,off_y_s)
        kdx, kdy = center_to_index(xd,TILE,off_x_d), center_to_index(yd,TILE,off_y_d)
        dst_centers.append((off_x_d+kdx*TILE, off_y_d+kdy*TILE))
        delta_idx.append((ksx-kdx, ksy-kdy))

    tree = cKDTree(dst_centers)
    for j in range(0,H,TILE):
        for i in range(0,W,TILE):
            if filled[j:j+TILE,i:i+TILE].all():
                continue
            _, idx = tree.query([i,j])
            dkx,dky = delta_idx[idx]
            ksx,ksy = i//TILE+dkx, j//TILE+dky
            if ksx<0 or ksy<0 or ksx*TILE>=W or ksy*TILE>=H:
                continue
            copy_region(src_mask,out_mask,
                        *idx_bounds(ksx,TILE,W),*idx_bounds(ksy,TILE,H),
                        i,i+TILE,j,j+TILE,
                        filled)

    # Stage 3 — TPS warping for remaining pixels
    remain = ~filled
    if remain.any():
        Fx,Fy = build_inverse_rbf(dst_pts,src_pts)
        gx,gy = np.meshgrid(np.arange(W),np.arange(H))
        map_x = Fx(gx,gy).astype(np.float32)
        map_y = Fy(gx,gy).astype(np.float32)
        tps = cv2.remap(src_mask,map_x,map_y,cv2.INTER_NEAREST)
        out_mask[remain] = tps[remain]

    # ===================== Visualization =====================
    overlay = (out_mask>0).astype(np.uint8)*255
    overlay = cv2.resize(overlay,(960,960),cv2.INTER_NEAREST)
    overlay = morphology.remove_small_objects(overlay.astype(bool),100).astype(np.uint8)*255
    overlay_d50 = cv2.dilate(overlay,np.ones((50,50),np.uint8),1)
    src_mask = cv2.resize(src_mask,(960,960),cv2.INTER_NEAREST)

    img00_vis = img00_raw.copy()
    img00_vis[src_mask>0] = (0.9*img00_vis[src_mask>0]+0.1*np.array([0,0,255])).astype(np.uint8)

    img11_vis = img11_raw.copy()
    img11_vis[overlay_d50>0] = (0.9*img11_vis[overlay_d50>0]+0.1*np.array([0,0,255])).astype(np.uint8)

    vis_pair = np.hstack([img00_vis, img11_vis])

    filled_big = cv2.resize(filled.astype(np.uint8),(img11_raw.shape[1],img11_raw.shape[0]),cv2.INTER_NEAREST).astype(bool)
    degraded_black = img11_raw.copy()
    degraded_black[~filled_big] = 0

    base = os.path.splitext(fname)[0]
    imwrite_unicode(os.path.join(out_transferred_clear_labels,base+".jpg"),overlay)
    imwrite_unicode(os.path.join(out_degraded_candidate_regions,base+".jpg"),overlay_d50)
    imwrite_unicode(os.path.join(out_degraded_candidate_visualization,base+".jpg"),vis_pair)
    imwrite_unicode(os.path.join(out_transferred_degraded_images,base+".jpg"),degraded_black)

print("Batch finished.")


In [ ]:
#Use the model trained on clear images in Step 2 to predict crack masks for the images in the **out_degraded_candidate_regions** folder, 
#and then take the intersection of the predictions with the candidate regions.

#The corresponding code can be found in the demo example in the **clear image annotation** directory.

In [ ]:
#Use the code below to crop images and construct the training dataset. 
#The training images are degraded images, and three types of labels are generated: 
#the pseudo-labels obtained by applying the model trained on clear images to degraded images, 
#the crack candidate regions in degraded images, and the reference labels from clear images annotation.
#The dataset format is detailed in the **training_dataset** directory.

path_img=''
from PIL import Image
import math
import numpy as np
import cv2
import os
from skimage import morphology
imgs = sorted([f for f in os.listdir(path_img1) if f.endswith('.jpg')])
i=0
def letterbox_unet(img, new_shape=(448, 448), color=(0,0,0), auto=False, scaleFill=False, scaleup=False):
    shape = img.shape[:2]  # current shape [height, width]
#     print(shape)
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)

    # Scale ratio (new / old)
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    if not scaleup:
        r = min(r, 1.0)

    # Compute padding
    ratio = r, r  
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]  # wh padding
    if auto:  # minimum rectangle
        dw, dh = np.mod(dw, 64), np.mod(dh, 64)  # wh padding
    elif scaleFill:  # stretch
        dw, dh = 0.0, 0.0
        new_unpad = (new_shape[1], new_shape[0])
        ratio = new_shape[1] / shape[1], new_shape[0] / shape[0]  # width, height ratios
    dw /= 2  # divide padding into 2 sides
    dh /= 2

    if shape[::-1] != new_unpad:  # resize
        img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)  # add border
    return img, ratio, (dw, dh)

for img in imgs:
    img = Image.open(path_img+img)
    img = np.asarray(img)
    if img.shape[0]<=448:
        img=img
    elif img.shape[0]>800:
        width=int(img.shape[1]*448*2/img.shape[0])
        dim=(width,448*2)
        img=cv2.resize(img,dim,interpolation=cv2.INTER_AREA)

    else:
        width=int(img.shape[1]*448*1/img.shape[0])
        dim=(width,448*1)
        img=cv2.resize(img,dim,interpolation=cv2.INTER_AREA)
    img_height = img.shape[0] 
    img_width = img.shape[1]
    input_size = [448,448]   
    input_width, input_height = input_size[0], input_size[1]

    x_num = math.ceil(img_width/input_width)
    y_num = math.ceil(img_height/input_height)
    normalization_map = np.zeros((img_height, img_width))
    # divide the image into patches with size of [input_height, input_width], and estimate cracks one patch by one patch
    for y in range(0, y_num):
        for x in range(0, x_num):
            x_coor = x*input_width
            y_coor = y*input_height
            if x_coor + input_width > img_width:
                x_coor1 = img_width
            else:
                x_coor1 = x_coor + input_width
            if y_coor + input_height > img_height:
                y_coor1 = img_height
            else:
                y_coor1 = y_coor + input_height
            if y == y_num-1 and y > 0:
                y_coor = y_coor1 - input_height
            if x == x_num -1 and x > 0:
                x_coor = x_coor1- input_width
            segment = img[y_coor:y_coor1, x_coor:x_coor1]
            if y_coor1-y_coor!= input_height or x_coor1-x_coor != input_width:
                segment, _, (dw, dh) = letterbox_unet(segment)
            segment = cv2.cvtColor(segment, cv2.COLOR_BGR2RGB)

            i+=1
            filename1=''+str(i).zfill(11)+'.jpg'
            cv2.imwrite(filename1,segment)
